In [ ]:
#Random Forest:

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.impute import SimpleImputer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
import pandas as pd
ds=pd.read_csv('/content/drive/My Drive/Infosys Internship/US_Accidents_March23.csv')

In [5]:
ds.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


In [9]:
target='Severity'
x=ds.drop(columns=[target])
y=ds[target]

In [10]:
categorical=x.select_dtypes(include=['object']).columns.tolist()
numerical=x.select_dtypes(include=['int64','float64','bool']).columns.tolist()

In [14]:
#numeric transformar pipeline
numeric_transformer=Pipeline(steps=[('imputer',SimpleImputer(strategy='median')),
                                    ('scaler',StandardScaler())])

#categorical transformer pipeline
categorical_transformer=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='constant',fill_value='missing')),
    ('onehot',OneHotEncoder(handle_unknown='ignore'))
    ])

#combine preprocessing for numeric and categorical
preprocessor=ColumnTransformer(transformers=[
    ('num',numeric_transformer,numerical),
    ('cat',categorical_transformer,categorical)
])

In [15]:
clf=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('classifier',RandomForestClassifier(random_state=42,n_estimators=100,n_jobs=-1))
])

In [18]:
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y
)

In [ ]:
clf.fit(x_train,y_train)

In [ ]:
y_pred=clf.predict(x_test)
print("Accuracy: ",accuracy_score(y-test,y_pred))
print("\nClassification Report:\n",classification_report(y_test,y_pred))
print("\nConfusion Matrix:\n",confusion_matrix(y_test,y_pred))

In [ ]:
##extracting feature names after OneHotEncoding
cat_features=clf.named_steps['classifier'].feature_importances_
indices=np.argsort(importance)[::-1]

plt.figure(figsize=(12,6))
plt.title('Feature importances from random forest classifier' )
plt.bar(range(len(importances)),importances[indices],align='center')
plt.xticks(range(len(importances)),all_features[indices],rotation=90)
plt.tight_layout()
plt.show()